# 🧘 Yoga Pose Similarity Search
### Transfer Learning + Milvus Vector Database + Flask Web App

This notebook:
1. **Installs** all required packages
2. **Downloads** the 107 Yoga Poses dataset from Kaggle
3. **Embeds** all images using a pre-trained ResNet50 (transfer learning)
4. **Stores** embeddings in a local Milvus vector database
5. **Queries** the database for similar poses
6. **Launches** a Flask web app (tunnelled via ngrok) so you can upload an image and see the most similar poses

> **Note:** Run cells top-to-bottom. Cells marked ⚙️ contain config you may want to edit.

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q milvus-lite pymilvus torch torchvision flask pyngrok pillow kaggle

---
## Cell 2 — ⚙️ Kaggle Authentication & Dataset Download

**Option A (recommended):** Upload your `kaggle.json` API key when prompted.  
**Option B:** Manually place `kaggle.json` in `/root/.kaggle/` before running.

Get your key at https://www.kaggle.com/settings → API → Create New Token.

In [ ]:
import os
from google.colab import files

# Upload kaggle.json
print("Please upload your kaggle.json file:")
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed ✓')

In [ ]:
# Download and unzip the dataset
!kaggle datasets download -d shrutisaxena/yoga-pose-image-classification-dataset -p /content/yogadata --unzip

# Locate the train directory
import glob
train_dirs = glob.glob('/content/yogadata/**/train', recursive=True)
DATASET_DIR = train_dirs[0] if train_dirs else '/content/yogadata'
print(f'Dataset root: {DATASET_DIR}')
print(f'Classes found: {len(os.listdir(DATASET_DIR))}')

---
## Cell 3 — ⚙️ Configuration

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
DB_PATH    = '/content/yoga_milvus.db'   # Milvus-lite local file
COLLECTION = 'yoga_embeddings'
EMB_DIM    = 2048                        # ResNet50 penultimate layer
IMG_SIZE   = 224
BATCH_SIZE = 64                          # images processed per batch
TOP_N      = 5                           # default results to return

print('Config set ✓')

---
## Cell 4 — Load Pre-trained ResNet50 Feature Extractor

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load ResNet50 pre-trained on ImageNet, strip the classification head
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])  # → (batch, 2048, 1, 1)
feature_extractor.eval().to(device)

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('ResNet50 feature extractor ready ✓')

---
## Cell 5 — Embedding Helper Function

In [ ]:
import numpy as np
from PIL import Image

def get_embedding(image_path: str) -> np.ndarray:
    """
    Returns an L2-normalized 2048-d embedding for a single image.
    Uses the pre-trained ResNet50 feature extractor (transfer learning).
    """
    img = Image.open(image_path).convert('RGB')
    tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = feature_extractor(tensor).squeeze().cpu().numpy()
    emb = emb / (np.linalg.norm(emb) + 1e-10)   # L2 normalise for cosine
    return emb.astype(np.float32)

print('get_embedding() defined ✓')

---
## Cell 6 — Build Milvus Vector Database

This cell scans every image in the dataset, computes its embedding, and inserts it into Milvus.  
Expect ~5-15 minutes depending on GPU availability and dataset size.

In [ ]:
from pymilvus import MilvusClient
import time

client = MilvusClient(DB_PATH)

# Re-create collection (drop if exists)
if client.has_collection(COLLECTION):
    client.drop_collection(COLLECTION)
    print(f'Dropped existing collection "{COLLECTION}"')

client.create_collection(
    collection_name=COLLECTION,
    dimension=EMB_DIM,
    metric_type='COSINE',
    auto_id=True,
)
print(f'Created collection "{COLLECTION}" (dim={EMB_DIM}, metric=COSINE) ✓')

# Walk dataset and embed all images
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.jpe'}
data_buffer = []
total, skipped = 0, 0
start = time.time()

labels = sorted(os.listdir(DATASET_DIR))
print(f'Processing {len(labels)} classes...')

for label in labels:
    class_dir = os.path.join(DATASET_DIR, label)
    if not os.path.isdir(class_dir):
        continue
    for fname in os.listdir(class_dir):
        ext = os.path.splitext(fname)[1].lower()
        if ext not in VALID_EXTS:
            continue
        fpath = os.path.join(class_dir, fname)
        try:
            emb = get_embedding(fpath)
            data_buffer.append({
                'vector':     emb,
                'label':      label,
                'image_path': fpath,
            })
            total += 1
        except Exception as e:
            skipped += 1

        # Insert in batches to avoid memory spikes
        if len(data_buffer) >= BATCH_SIZE:
            client.insert(collection_name=COLLECTION, data=data_buffer)
            data_buffer = []

# Insert remaining
if data_buffer:
    client.insert(collection_name=COLLECTION, data=data_buffer)

elapsed = time.time() - start
print(f'\n✅ Inserted {total} embeddings | Skipped {skipped} | Time: {elapsed:.1f}s')
client.close()

---
## Cell 7 — Similarity Search Function

In [ ]:
def find_similar(image_path: str, top_n: int = TOP_N) -> list:
    """
    Query the Milvus collection for the top_n most similar yoga pose images.
    Returns a list of dicts: [{score, label, image_path}, ...]
    """
    query_emb = get_embedding(image_path)
    client = MilvusClient(DB_PATH)
    results = client.search(
        collection_name=COLLECTION,
        data=[query_emb],
        limit=top_n,
        output_fields=['label', 'image_path'],
        search_params={'metric_type': 'COSINE'},
    )
    client.close()
    hits = []
    for hit in results[0]:
        hits.append({
            'score':      round(float(hit['distance']), 4),
            'label':      hit['entity']['label'],
            'image_path': hit['entity']['image_path'],
        })
    return hits

print('find_similar() defined ✓')

---
## Cell 8 — Quick Inline Test (no web app needed)

Picks a random image from the dataset, queries Milvus, and displays results inline.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Pick a random test image
all_images = [
    os.path.join(root, f)
    for root, _, files in os.walk(DATASET_DIR)
    for f in files
    if os.path.splitext(f)[1].lower() in {'.jpg', '.jpeg', '.png', '.jpe'}
]
query_path = random.choice(all_images)
query_label = os.path.basename(os.path.dirname(query_path))
print(f'Query image: {query_path}  (class: {query_label})')

hits = find_similar(query_path, top_n=5)

# Display
fig, axes = plt.subplots(1, 6, figsize=(18, 4))
axes[0].imshow(mpimg.imread(query_path))
axes[0].set_title(f'QUERY\n{query_label}', fontsize=9, color='blue')
axes[0].axis('off')

for i, hit in enumerate(hits, start=1):
    axes[i].imshow(mpimg.imread(hit['image_path']))
    axes[i].set_title(f"{hit['label']}\nscore={hit['score']}", fontsize=8)
    axes[i].axis('off')

plt.suptitle('Yoga Pose Similarity Search — Top 5 Results', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

---
## Cell 9 — ⚙️ ngrok Authentication

Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken  
Paste it below to enable the public tunnel for the Flask app.

In [ ]:
NGROK_TOKEN = ''   # ← paste your token here

if NGROK_TOKEN:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_TOKEN
    print('ngrok token set ✓')
else:
    print('⚠️  No ngrok token set — skip to Cell 11 to use the inline uploader instead.')

---
## Cell 10 — Write Flask App Files

In [ ]:
import os

os.makedirs('/content/flask_app/templates', exist_ok=True)
os.makedirs('/content/flask_app/static/uploads', exist_ok=True)

# ── app.py ───────────────────────────────────────────────────────────────────
APP_PY = '''
import os, shutil, sys
sys.path.insert(0, "/content")
from flask import Flask, request, render_template, send_from_directory
from pymilvus import MilvusClient
import torch, torchvision.transforms as T, numpy as np
from torchvision import models
from PIL import Image

DB_PATH    = "/content/yoga_milvus.db"
COLLECTION = "yoga_embeddings"
EMB_DIM    = 2048
IMG_SIZE   = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
fe = torch.nn.Sequential(*list(resnet.children())[:-1])
fe.eval().to(device)

preprocess = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def get_embedding(path):
    img = Image.open(path).convert("RGB")
    t = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        e = fe(t).squeeze().cpu().numpy()
    e = e / (np.linalg.norm(e) + 1e-10)
    return e.astype(np.float32)

def find_similar(path, top_n=5):
    q = get_embedding(path)
    c = MilvusClient(DB_PATH)
    res = c.search(
        collection_name=COLLECTION, data=[q], limit=top_n,
        output_fields=["label","image_path"],
        search_params={"metric_type":"COSINE"})
    c.close()
    return [{"score": round(float(h["distance"]),4),
             "label": h["entity"]["label"],
             "image_path": h["entity"]["image_path"]} for h in res[0]]

app = Flask(__name__, template_folder="templates", static_folder="static")
UPLOAD = "static/uploads"
os.makedirs(UPLOAD, exist_ok=True)
ALLOWED = {"jpg","jpeg","png"}

@app.route("/", methods=["GET","POST"])
def index():
    results, query_image = [], None
    if request.method == "POST":
        f = request.files.get("image")
        if f and f.filename.rsplit(".",1)[-1].lower() in ALLOWED:
            save = os.path.join(UPLOAD, f.filename)
            f.save(save)
            query_image = f.filename
            top_n = int(request.form.get("top_n", 5))
            for hit in find_similar(save, top_n):
                dest_name = hit["image_path"].replace("/","_")
                dest = os.path.join("static/uploads", dest_name)
                if not os.path.exists(dest):
                    shutil.copy(hit["image_path"], dest)
                results.append({"score":hit["score"],"label":hit["label"],"filename":dest_name})
    return render_template("index.html", query_image=query_image, results=results)

if __name__ == "__main__":
    app.run(port=5000)
'''

# ── templates/index.html ─────────────────────────────────────────────────────
INDEX_HTML = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"/>
  <meta name="viewport" content="width=device-width,initial-scale=1"/>
  <title>Yoga Pose Search</title>
  <style>
    body{font-family:Arial,sans-serif;max-width:960px;margin:40px auto;padding:0 20px;background:#f5f5f5}
    h1{color:#4a4a8a}
    form{background:#fff;padding:20px;border-radius:8px;box-shadow:0 2px 6px rgba(0,0,0,.1)}
    button{background:#4a4a8a;color:#fff;padding:10px 24px;border:none;border-radius:5px;cursor:pointer;font-size:1rem}
    button:hover{background:#36367a}
    .grid{display:flex;flex-wrap:wrap;gap:16px;margin-top:16px}
    .card{background:#fff;border-radius:8px;padding:12px;box-shadow:0 2px 6px rgba(0,0,0,.1);text-align:center;width:160px}
    .card img{width:140px;height:140px;object-fit:cover;border-radius:4px}
    .label{font-weight:bold;margin-top:8px;color:#4a4a8a;text-transform:capitalize;font-size:.9rem}
    .score{font-size:.8rem;color:#777}
    .qimg{width:200px;height:200px;object-fit:cover;border-radius:8px;border:3px solid #4a4a8a}
    .section{margin-top:28px}
  </style>
</head>
<body>
<h1>&#129336; Yoga Pose Similarity Search</h1>
<form method="POST" enctype="multipart/form-data">
  <label><strong>Upload a yoga pose image:</strong></label><br/>
  <input type="file" name="image" accept="image/*" required/><br/><br/>
  <label><strong>Results to return:</strong>
    <input type="number" name="top_n" value="5" min="1" max="20" style="width:60px;margin-left:8px"/>
  </label><br/><br/>
  <button type="submit">Find Similar Poses</button>
</form>
{% if query_image %}
<div class="section">
  <h2>Your Upload</h2>
  <img class="qimg" src="/static/uploads/{{ query_image }}" alt="query"/>
</div>
{% endif %}
{% if results %}
<div class="section">
  <h2>Most Similar Poses</h2>
  <div class="grid">
    {% for r in results %}
    <div class="card">
      <img src="/static/uploads/{{ r.filename }}" alt="{{ r.label }}"/>
      <div class="label">{{ r.label }}</div>
      <div class="score">score: {{ r.score }}</div>
    </div>
    {% endfor %}
  </div>
</div>
{% endif %}
</body>
</html>
'''

with open('/content/flask_app/app.py', 'w') as fh:
    fh.write(APP_PY)

with open('/content/flask_app/templates/index.html', 'w') as fh:
    fh.write(INDEX_HTML)

print('Flask app files written ✓')

---
## Cell 11 — Launch Flask App (with ngrok public URL)

After running this cell, click the public URL that appears to open the web app.

> **To stop the server**, interrupt the kernel (Runtime → Interrupt execution).

In [ ]:
import subprocess, time, threading
from pyngrok import ngrok

os.chdir('/content/flask_app')

# Start Flask in a background thread
def run_flask():
    subprocess.run(['python', 'app.py'])

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(3)  # give Flask time to start

# Open ngrok tunnel
public_url = ngrok.connect(5000)
print(f'\n🌐 Web App URL: {public_url}')
print('Open the URL above in your browser to use the yoga pose search app.')
print('(Interrupt the kernel to stop the server)')

# Keep alive
while True:
    time.sleep(60)

---
## Cell 12 — (Alternative) Inline Upload & Search — No ngrok needed

If you don't want to use ngrok, use this cell instead of Cells 9–11.  
Upload an image from your computer and see results plotted inline.

In [ ]:
from google.colab import files
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print('Upload a yoga pose image:')
uploaded = files.upload()

for fname, data in uploaded.items():
    query_path = f'/content/{fname}'
    with open(query_path, 'wb') as f:
        f.write(data)

    print(f'\nSearching for poses similar to: {fname}')
    hits = find_similar(query_path, top_n=5)

    fig, axes = plt.subplots(1, 6, figsize=(20, 4))
    axes[0].imshow(mpimg.imread(query_path))
    axes[0].set_title('QUERY', fontsize=10, color='blue', fontweight='bold')
    axes[0].axis('off')

    for i, hit in enumerate(hits, start=1):
        try:
            axes[i].imshow(mpimg.imread(hit['image_path']))
        except Exception:
            pass
        axes[i].set_title(f"{hit['label']}\nscore={hit['score']}", fontsize=8)
        axes[i].axis('off')

    plt.suptitle(f'Top 5 Similar Yoga Poses to "{fname}"', fontsize=12)
    plt.tight_layout()
    plt.show()

    print('\nTop results:')
    for h in hits:
        print(f"  [{h['score']:.4f}]  {h['label']}  →  {h['image_path']}")

---
## Cell 13 — Inspect the Vector Database

In [ ]:
client = MilvusClient(DB_PATH)
stats = client.get_collection_stats(COLLECTION)
print(f'Collection : {COLLECTION}')
print(f'Total rows : {stats["row_count"]}')
print(f'Embedding dim: {EMB_DIM}')
print(f'Metric type  : COSINE')
client.close()